In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import os
import re
import pickle
from dotenv import load_dotenv

import torch
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
query = "Yaya kuralları nelerdir?"
top_k = 5

### Dense Search

In [6]:
client = QdrantClient("localhost", port=6333)

In [7]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
embed_model_name = "intfloat/multilingual-e5-small"
embed_model = SentenceTransformer(embed_model_name, token=hf_token, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8147.38it/s]


In [8]:
query_emb = embed_model.encode([query])[0]
dense_results = client.query_points(
    collection_name="legal_docs", query=query_emb.tolist(), limit=top_k
).points
dense_results = [(str(r.payload["chunk_id"]), r.score) for r in dense_results]

### BM25 Search

In [9]:
# Clean the texts
def tokenize_tr(text: str) -> list[str]:
    text = text.replace("İ", "i").replace("I", "ı").lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)

    return text.split()

In [10]:
with open("../data/processed/bm25_index.pkl", "rb") as f:
    bm25_index = pickle.load(f)

In [11]:
bm25 = bm25_index["bm25"]
chunks = bm25_index["chunks"]

In [12]:
tokenized_query = tokenize_tr(query)
scores = bm25.get_scores(tokenized_query)
scored = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]

sparse_results = [(chunks[idx]["chunk_id"], float(score)) for idx, score in scored]

## Hybrid Search

In [13]:
k = 60

In [14]:
rrf_scores = {}

# Dense sonuçlarını ekle
for rank, (chunk_id, _) in enumerate(dense_results):
    rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank + 1)

# BM25 sonuçlarını ekle
# BM25 chunk dict döndürüyor, chunk_id'yi metadata'dan al
for rank, (chunk_id, _) in enumerate(sparse_results):
    rrf_scores[chunk_id] = rrf_scores.get(chunk_id, 0) + 1 / (k + rank + 1)

# Sırala
sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]

# chunk_id → chunk dict mapping oluştur
id_to_chunk = {c["chunk_id"]: c for c in chunks}

In [15]:
hybrid_results = [
    {**id_to_chunk[cid], "rrf_score": rrf_scores[cid]}
    for cid in sorted_ids
    if cid in id_to_chunk
]

In [16]:
dense_results

[('trafik_kanunu_72', 0.8602816),
 ('trafik_kanunu_57', 0.8485191),
 ('trafik_kanunu_4', 0.84808743),
 ('trafik_kanunu_73', 0.84501696),
 ('trafik_kanunu_59', 0.84170336)]

In [17]:
sparse_results

[('trafik_kanunu_55', 6.952230564235439),
 ('trafik_kanunu_72', 4.942037286419887),
 ('trafik_kanunu_57', 4.884655848195635),
 ('trafik_kanunu_78', 3.845667899199946),
 ('trafik_kanunu_48', 3.808803902672261)]

In [18]:
rrf_scores

{'trafik_kanunu_72': 0.03252247488101534,
 'trafik_kanunu_57': 0.03200204813108039,
 'trafik_kanunu_4': 0.015873015873015872,
 'trafik_kanunu_73': 0.015625,
 'trafik_kanunu_59': 0.015384615384615385,
 'trafik_kanunu_55': 0.01639344262295082,
 'trafik_kanunu_78': 0.015625,
 'trafik_kanunu_48': 0.015384615384615385}

In [19]:
hybrid_results

[{'chunk_id': 'trafik_kanunu_72',
  'text': '68 – Yayaların uyacakları kurallar aşağıda belirtilmiştir.\n58 Anayasa Mahkemesinin 14/3/2019 tarihli ve E.: 2019/1, K.: 2019/14 sayılı Kararı ile, bu fıkranın\nüçüncü cümlesi “Sürücünün araç sahibi olmadığı hâl” yönünden iptal edilmiştir.\n59 12/2/2026 tarihli ve 7574 sayılı Kanunun 21 inci maddesiyle bu fıkrada yer alan “5.010” ibaresi\n“140.000” şeklinde değiştirilmiştir.\n60 12/2/2026 tarihli ve 7574 sayılı Kanunun 21 inci maddesiyle bu fıkrada yer alan “Belgesi iptal\nedilenlerin” ibaresi “Sürücü belgesi iptal edilenlerin” şeklinde değiştirilmiş, fıkraya “eğitime\nbaşlayabilmeleri için” ibaresinden sonra gelmek üzere “sürücü belgesinin iptal edildiği tarihten\nitibaren iptal nedenlerinde yer alan geri alma süreleri kadar zamanın geçmiş olması, bu Kanun\nkapsamında verilen idari para cezalarının tamamının tahsil edilmiş olması ve” ibaresi eklenmştir.a) Yayalar, aşağıda sayılan haller dışında, taşıt yolu bitişiğinde ve yakınında yaya\nyol